### Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import json
import cv2
import os
from PIL import Image
from typing import Optional, Tuple, Dict
import shutil
import random

### Models

#### Sanity Check Base CNN

In [2]:
class BaseCNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(BaseCNN, self).__init__()

        # Conv layers (same as before)
        self.conv1 = nn.Conv2d(4, 32, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        # FIXED: Add adaptive pooling
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        # FC layers
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(512, 512)  # FIXED: 512 input (not 1M+)
        self.fc2 = nn.Linear(512, 256)
        self.fc_conf = nn.Linear(256, 1)
        self.fc_vec = nn.Linear(256, 6)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.bn3(self.conv3(x)))
        x = torch.relu(self.bn4(self.conv4(x)))
        x = torch.relu(self.bn5(self.conv5(x)))

        # FIXED: Apply adaptive pooling
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)

        confidence = torch.sigmoid(self.fc_conf(x))
        vector = self.fc_vec(x)
        return confidence, vector

#### YOLO-based CNN

In [3]:
"""
YOLO-Inspired CNN Model for 3D Pointing Detection
Uses CSPDarknet backbone architecture for improved feature extraction
"""
class ConvBlock(nn.Module):
    """Basic convolutional block with Conv, BatchNorm, and activation"""

    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, activation='silu'):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

        if activation == 'silu':
            self.act = nn.SiLU(inplace=True)
        elif activation == 'relu':
            self.act = nn.ReLU(inplace=True)
        elif activation == 'leaky':
            self.act = nn.LeakyReLU(0.1, inplace=True)
        else:
            self.act = nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    """Residual block used in Darknet"""

    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        half_channels = channels // 2
        self.conv1 = ConvBlock(channels, half_channels, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(half_channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + residual


class CSPBlock(nn.Module):
    """Cross Stage Partial block from CSPNet"""

    def __init__(self, in_channels, out_channels, num_blocks=1, shortcut=True):
        super(CSPBlock, self).__init__()
        half_channels = out_channels // 2

        # Split
        self.conv1 = ConvBlock(in_channels, half_channels, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(in_channels, half_channels, kernel_size=1, padding=0)

        # Residual blocks
        self.blocks = nn.Sequential(
            *[ResidualBlock(half_channels) for _ in range(num_blocks)]
        )

        # Merge
        self.conv3 = ConvBlock(half_channels, half_channels, kernel_size=1, padding=0)
        self.conv4 = ConvBlock(out_channels, out_channels, kernel_size=1, padding=0)

        self.shortcut = shortcut

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x)

        x1 = self.blocks(x1)
        x1 = self.conv3(x1)

        x = torch.cat([x1, x2], dim=1)
        x = self.conv4(x)

        return x


class SPPBlock(nn.Module):
    """Spatial Pyramid Pooling block"""

    def __init__(self, in_channels, out_channels, kernel_sizes=[5, 9, 13]):
        super(SPPBlock, self).__init__()
        self.kernel_sizes = kernel_sizes
        self.conv1 = ConvBlock(in_channels, in_channels // 2, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(in_channels // 2 * (len(kernel_sizes) + 1), out_channels, kernel_size=1, padding=0)

    def forward(self, x):
        x = self.conv1(x)
        features = [x]

        for kernel_size in self.kernel_sizes:
            padding = kernel_size // 2
            pooled = F.max_pool2d(x, kernel_size, stride=1, padding=padding)
            features.append(pooled)

        x = torch.cat(features, dim=1)
        x = self.conv2(x)
        return x


class YOLOBackbone(nn.Module):
    """
    YOLO-inspired backbone for 3D pointing detection
    Uses CSPDarknet architecture
    """

    def __init__(self, input_channels=4, base_channels=64, depth_multiple=1.0, width_multiple=1.0):
        super(YOLOBackbone, self).__init__()

        # Calculate channel sizes based on width_multiple
        def make_divisible(x, divisor=8):
            return int((x * width_multiple + divisor / 2) // divisor * divisor)

        # Stem
        self.stem = ConvBlock(input_channels, make_divisible(base_channels), kernel_size=6, stride=2, padding=2)

        # Stage 1
        self.stage1 = nn.Sequential(
            ConvBlock(make_divisible(base_channels), make_divisible(base_channels * 2), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 2), make_divisible(base_channels * 2), num_blocks=max(round(3 * depth_multiple), 1))
        )

        # Stage 2
        self.stage2 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 2), make_divisible(base_channels * 4), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 4), make_divisible(base_channels * 4), num_blocks=max(round(6 * depth_multiple), 1))
        )

        # Stage 3
        self.stage3 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 4), make_divisible(base_channels * 8), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 8), make_divisible(base_channels * 8), num_blocks=max(round(9 * depth_multiple), 1))
        )

        # Stage 4
        self.stage4 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 8), make_divisible(base_channels * 16), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 16), make_divisible(base_channels * 16), num_blocks=max(round(3 * depth_multiple), 1))
        )

        # SPP
        self.spp = SPPBlock(make_divisible(base_channels * 16), make_divisible(base_channels * 16))

        self.final_channels = make_divisible(base_channels * 16)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.spp(x)
        return x


class YOLOPointingNet(nn.Module):
    """
    Complete YOLO-based network for 3D pointing detection

    Input: 4-channel image (RGB + Depth) of shape (B, 4, H, W)
    Output:
        - confidence: (B, 1) - probability of pointing gesture
        - vector: (B, 6) - two 3D points (wrist position + direction vector)
    """

    def __init__(self, input_height=720, input_width=1280, input_channels=4,
                 depth_multiple=0.33, width_multiple=0.5, dropout_rate=0.2):
        """
        Args:
            input_height: Input image height
            input_width: Input image width
            input_channels: Number of input channels (4 for RGB-D)
            depth_multiple: Depth scaling factor (0.33 for small, 0.67 for medium, 1.0 for large)
            width_multiple: Width scaling factor (0.5 for small, 0.75 for medium, 1.0 for large)
            dropout_rate: Dropout probability
        """
        super(YOLOPointingNet, self).__init__()

        # Backbone
        self.backbone = YOLOBackbone(input_channels, base_channels=64,
                                     depth_multiple=depth_multiple,
                                     width_multiple=width_multiple)

        # Calculate flattened size
        # After 5 stride-2 operations: H/32 x W/32
        self.flat_size = self.backbone.final_channels * (input_height // 32) * (input_width // 32)

        # Global Average Pooling (alternative to flattening)
        self.use_gap = True

        if self.use_gap:
            self.gap = nn.AdaptiveAvgPool2d(1)
            fc_input_size = self.backbone.final_channels
        else:
            fc_input_size = self.flat_size

        # Head network
        self.dropout = nn.Dropout(dropout_rate)

        # Shared feature layers
        self.fc1 = nn.Linear(fc_input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)

        # Classification head (pointing vs not pointing)
        self.fc_conf = nn.Linear(256, 1)

        # Regression head (6 values: 3 for wrist position, 3 for direction vector)
        self.fc_vec1 = nn.Linear(256, 128)
        self.fc_vec2 = nn.Linear(128, 6)

        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        """
        Forward pass

        Args:
            x: Input tensor of shape (B, 4, H, W)

        Returns:
            confidence: (B, 1) - sigmoid-activated confidence score
            vector: (B, 6) - predicted 3D vector components
        """
        # Backbone feature extraction
        x = self.backbone(x)

        # Pooling and flattening
        if self.use_gap:
            x = self.gap(x)
            x = x.view(x.size(0), -1)
        else:
            x = x.view(x.size(0), -1)

        # Shared features
        x = F.silu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.silu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        # Classification head
        confidence = torch.sigmoid(self.fc_conf(x))

        # Regression head
        vector = F.silu(self.fc_vec1(x))
        vector = self.fc_vec2(vector)

        return confidence, vector


### Dataset

In [4]:
class PointingDataset(Dataset):
    """
    Dataset for 4D (RGB-D) pointing gesture recognition.

    Data format per sample:
    - RGB image: {timestamp}.jpg (1920x1080x3, BGR uint8)
    - Depth image: {timestamp}.npy (1080x1920, uint16, millimeters)
    - Label: {timestamp}.txt
        - Line 1: label (0 or 1)
        - Lines 2-7 (if label==1): 6 floats (wrist_x, wrist_y, wrist_z, dir_x, dir_y, dir_z)
    """

    def __init__(self, data_dir: str, transform: Optional[callable] = None):
        """
        Args:
            data_dir: Path to directory containing .jpg, .npy, and .txt files
            transform: Optional transform to apply to the image
        """
        self.data_dir = data_dir
        self.transform = transform

        # find all .jpg files (each represents a complete sample)
        self.samples = []
        if os.path.exists(data_dir):
            for filename in os.listdir(data_dir):
                if filename.endswith('.jpg'):
                    base_name = filename[:-4]  # remove .jpg extension
                    sample = {
                        'base_name': base_name,
                        'image_path': os.path.join(data_dir, filename),
                        'depth_path': os.path.join(data_dir, base_name + '.npy'),
                        'label_path': os.path.join(data_dir, base_name + '.txt')
                    }
                    self.samples.append(sample)

        print(f"Found {len(self.samples)} samples in {data_dir}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, dict]:
        """
        Returns:
            rgbd_image: torch.Tensor of shape (4, H, W) - RGB-D concatenated
            label_dict: dict containing {
                'is_pointing': int (0 or 1),
                'wrist_coords': torch.Tensor (3,) or None,
                'pointing_vector': torch.Tensor (3,) or None
            }
        """
        sample = self.samples[idx]

        # Load RGB image (BGR -> RGB)
        bgr_img = cv2.imread(sample['image_path'])
        rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        rgb_img = rgb_img.astype(np.float32) / 255.0  # Normalize to [0, 1]
        rgb_img = np.transpose(rgb_img, (2, 0, 1))  # (H, W, C) -> (C, H, W)

        # Load depth image
        depth_img = np.load(sample['depth_path'])
        depth_img = depth_img.astype(np.float32) / 1000.0  # Convert mm to meters
        depth_img = np.clip(depth_img, 0, 10.0)  # Clip to 0-10m range
        depth_img = np.expand_dims(depth_img, axis=0)  # Add channel dimension (1, H, W)

        # Concatenate RGB + D to get 4D image
        rgbd_image = np.concatenate([rgb_img, depth_img], axis=0)  # (4, H, W)

        # Apply transform if provided
        if self.transform:
            rgbd_image = self.transform(rgbd_image)

        # Load labels
        label_dict = self._load_label(sample['label_path'])

        # Convert to torch tensors
        rgbd_image = torch.from_numpy(rgbd_image).float()
        if label_dict['wrist_coords'] is not None:
            label_dict['wrist_coords'] = torch.from_numpy(label_dict['wrist_coords']).float()
        if label_dict['pointing_vector'] is not None:
            label_dict['pointing_vector'] = torch.from_numpy(label_dict['pointing_vector']).float()

        return rgbd_image, label_dict

    def _load_label(self, label_path: str) -> dict:
        """Load label from .txt file"""
        with open(label_path, 'r') as f:
            lines = f.readlines()

        label = int(lines[0].strip())

        result = {
            'is_pointing': label
        }

        if label == 1 and len(lines) >= 7:
            # Parse wrist coordinates and pointing vector
            wrist_coords = np.array([
                float(lines[1].strip()),
                float(lines[2].strip()),
                float(lines[3].strip())
            ])
            pointing_vector = np.array([
                float(lines[4].strip()),
                float(lines[5].strip()),
                float(lines[6].strip())
            ])
            result['wrist_coords'] = wrist_coords
            result['pointing_vector'] = pointing_vector
        else:
            result['wrist_coords'] = None
            result['pointing_vector'] = None

        return result

### Data Splitting

In [5]:
def split_pointing_data(data_dir, output_dir, train_ratio=0.7,
                       val_ratio=0.15, test_ratio=0.15,
                       stratify=True, seed=42):
    """Split your data into train/val/test"""

    # Find samples
    samples = []
    for jpg in Path(data_dir).glob("*.jpg"):
        base = jpg.stem
        npy = Path(data_dir) / f"{base}.npy"
        txt = Path(data_dir) / f"{base}.txt"
        if npy.exists() and txt.exists():
            samples.append({'jpg': jpg, 'npy': npy, 'txt': txt})

    print(f"Found {len(samples)} samples")

    # Get labels
    def get_label(txt):
        with open(txt) as f:
            return int(f.readline().strip())

    # Stratified split
    if stratify:
        pointing = [s for s in samples if get_label(s['txt']) == 1]
        not_pointing = [s for s in samples if get_label(s['txt']) == 0]

        random.seed(seed)
        random.shuffle(pointing)
        random.shuffle(not_pointing)

        def split(lst):
            n = len(lst)
            t = int(n * train_ratio)
            v = t + int(n * val_ratio)
            return lst[:t], lst[t:v], lst[v:]

        p_t, p_v, p_te = split(pointing)
        np_t, np_v, np_te = split(not_pointing)

        train = p_t + np_t
        val = p_v + np_v
        test = p_te + np_te

        for s in [train, val, test]:
            random.shuffle(s)
    else:
        random.seed(seed)
        random.shuffle(samples)
        n = len(samples)
        t = int(n * train_ratio)
        v = t + int(n * val_ratio)
        train, val, test = samples[:t], samples[t:v], samples[v:]

    # Copy files
    for name, split in [('train', train), ('val', val), ('test', test)]:
        d = Path(output_dir) / name
        d.mkdir(parents=True, exist_ok=True)
        for s in split:
            for k in ['jpg', 'npy', 'txt']:
                shutil.copy2(s[k], d / s[k].name)
        print(f"✓ {name}: {len(split)} samples")

    return {'train': len(train), 'val': len(val), 'test': len(test)}

### Training

In [ ]:
class PointingLoss(nn.Module):
    """
    Combined loss for pointing detection
    Includes binary cross-entropy for classification and MSE for regression
    """

    def __init__(self, alpha=1.0, beta=1.0, vector_weight_threshold=0.5):
        """
        Args:
            alpha: Weight for classification loss
            beta: Weight for regression loss
            vector_weight_threshold: Only compute vector loss if confidence > threshold
        """
        super(PointingLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.vector_weight_threshold = vector_weight_threshold

        self.bce_loss = nn.BCELoss()
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_confidence, pred_vector, true_confidence, true_vector):
        """
        Compute combined loss

        Args:
            pred_confidence: Predicted confidence (B, 1)
            pred_vector: Predicted vector (B, 6)
            true_confidence: True confidence (B, 1)
            true_vector: True vector (B, 6)
        """
        # Classification loss
        conf_loss = self.bce_loss(pred_confidence, true_confidence)

        # Regression loss (only for positive samples)
        # Create mask for samples where person is pointing
        pointing_mask = (true_confidence > self.vector_weight_threshold).squeeze()

        if pointing_mask.sum() > 0:
            # Compute MSE only for pointing samples
            vec_loss = self.mse_loss(pred_vector[pointing_mask], true_vector[pointing_mask])
        else:
            vec_loss = torch.tensor(0.0, device=pred_vector.device)

        # Combined loss
        total_loss = self.alpha * conf_loss + self.beta * vec_loss

        return total_loss, conf_loss, vec_loss


class AngularLoss(nn.Module):
    """
    Alternative loss that measures angular error for direction vector
    This can be more appropriate for directional predictions
    """

    def __init__(self, alpha=1.0, beta=1.0):
        super(AngularLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.bce_loss = nn.BCELoss()
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_confidence, pred_vector, true_confidence, true_vector):
        # Classification loss
        conf_loss = self.bce_loss(pred_confidence, true_confidence)

        # Only compute vector loss for pointing samples
        pointing_mask = (true_confidence > 0.5).squeeze()

        if pointing_mask.sum() > 0:
            # Split into position and direction
            pred_pos = pred_vector[pointing_mask, :3]
            pred_dir = pred_vector[pointing_mask, 3:]
            true_pos = true_vector[pointing_mask, :3]
            true_dir = true_vector[pointing_mask, 3:]

            # Position loss (MSE)
            pos_loss = self.mse_loss(pred_pos, true_pos)

            # Direction loss (cosine similarity)
            # Normalize vectors
            pred_dir_norm = F.normalize(pred_dir, p=2, dim=1)
            true_dir_norm = F.normalize(true_dir, p=2, dim=1)

            # Cosine similarity loss (1 - cos_sim)
            cos_sim = (pred_dir_norm * true_dir_norm).sum(dim=1).mean()
            dir_loss = 1 - cos_sim
            F.cosine_similarity()

            vec_loss = pos_loss + dir_loss
        else:
            vec_loss = torch.tensor(0.0, device=pred_vector.device)

        total_loss = self.alpha * conf_loss + self.beta * vec_loss

        return total_loss, conf_loss, vec_loss

In [ ]:
def collate_pointing_batch(batch):
    """
    Custom collate function to handle dict-based labels

    Args:
        batch: List of (rgbd_image, label_dict) tuples

    Returns:
        rgbd_batch: Stacked tensor of RGB-D images
        label_dict_batch: Dictionary with batched labels
    """
    rgbd_images = []
    is_pointing_list = []
    wrist_coords_list = []
    pointing_vector_list = []

    for rgbd, label_dict in batch:
        rgbd_images.append(rgbd)
        is_pointing_list.append(label_dict['is_pointing'])
        wrist_coords_list.append(label_dict['wrist_coords'])
        pointing_vector_list.append(label_dict['pointing_vector'])

    # Stack RGB-D images
    rgbd_batch = torch.stack(rgbd_images, dim=0)

    # Create batched label dict
    label_dict_batch = {
        'is_pointing': is_pointing_list,
        'wrist_coords': wrist_coords_list,
        'pointing_vector': pointing_vector_list
    }

    return rgbd_batch, label_dict_batch

def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()

    total_loss = 0.0
    total_conf_loss = 0.0
    total_vec_loss = 0.0

    for batch_idx, (rgbd, label_dict) in enumerate(dataloader):
        # Move to device
        rgbd = rgbd.to(device)

        # Prepare labels
        batch_size = rgbd.size(0)
        confidence = torch.zeros(batch_size, 1, dtype=torch.float32, device=device)
        vector = torch.zeros(batch_size, 6, dtype=torch.float32, device=device)

        for i in range(batch_size):
            confidence[i, 0] = float(label_dict['is_pointing'][i])

            if label_dict['wrist_coords'][i] is not None:
                wrist = label_dict['wrist_coords'][i]
                pointing = label_dict['pointing_vector'][i]
                vector[i, :3] = wrist.to(device)
                vector[i, 3:] = pointing.to(device)

        # Forward pass
        pred_confidence, pred_vector = model(rgbd)

        # Compute loss
        loss, conf_loss, vec_loss = criterion(pred_confidence, pred_vector, confidence, vector)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate losses
        total_loss += loss.item()
        total_conf_loss += conf_loss.item()
        total_vec_loss += vec_loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(dataloader)}: "
                  f"Loss={loss.item():.4f}, Conf={conf_loss.item():.4f}, Vec={vec_loss.item():.4f}")

    # Average losses
    avg_loss = total_loss / len(dataloader)
    avg_conf_loss = total_conf_loss / len(dataloader)
    avg_vec_loss = total_vec_loss / len(dataloader)

    return avg_loss, avg_conf_loss, avg_vec_loss


def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()

    total_loss = 0.0
    total_conf_loss = 0.0
    total_vec_loss = 0.0

    # Metrics
    correct = 0
    total = 0

    with torch.no_grad():
        for rgbd, label_dict in dataloader:
            # Move to device
            rgbd = rgbd.to(device)

            # Prepare labels
            batch_size = rgbd.size(0)
            confidence = torch.zeros(batch_size, 1, dtype=torch.float32, device=device)
            vector = torch.zeros(batch_size, 6, dtype=torch.float32, device=device)

            for i in range(batch_size):
                confidence[i, 0] = float(label_dict['is_pointing'][i])

                if label_dict['wrist_coords'][i] is not None:
                    wrist = label_dict['wrist_coords'][i]
                    pointing = label_dict['pointing_vector'][i]
                    vector[i, :3] = wrist.to(device)
                    vector[i, 3:] = pointing.to(device)

            # Forward pass
            pred_confidence, pred_vector = model(rgbd)

            # Compute loss
            loss, conf_loss, vec_loss = criterion(pred_confidence, pred_vector, confidence, vector)

            # Accumulate losses
            total_loss += loss.item()
            total_conf_loss += conf_loss.item()
            total_vec_loss += vec_loss.item()

            # Classification accuracy
            pred_class = (pred_confidence > 0.5).float()
            true_class = (confidence > 0.5).float()
            correct += (pred_class == true_class).sum().item()
            total += confidence.size(0)

    # Average losses and accuracy
    avg_loss = total_loss / len(dataloader)
    avg_conf_loss = total_conf_loss / len(dataloader)
    avg_vec_loss = total_vec_loss / len(dataloader)
    accuracy = 100.0 * correct / total

    return avg_loss, avg_conf_loss, avg_vec_loss, accuracy

def train_model(model, train_loader, val_loader, num_epochs=50, lr=1e-4, device='cuda'):
    """
    Complete training loop

    Args:
        model: PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Number of training epochs
        lr: Learning rate
        device: Device to train on
    """
    model = model.to(device)

    # Loss and optimizer
    criterion = PointingLoss(alpha=1.0, beta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                       patience=5)

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print("-" * 50)

        # Train
        train_loss, train_conf_loss, train_vec_loss = train_epoch(
            model, train_loader, criterion, optimizer, device
        )

        print(f"Train Loss: {train_loss:.4f} (Conf: {train_conf_loss:.4f}, Vec: {train_vec_loss:.4f})")

        # Validate
        val_loss, val_conf_loss, val_vec_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        print(f"Val Loss: {val_loss:.4f} (Conf: {val_conf_loss:.4f}, Vec: {val_vec_loss:.4f})")
        print(f"Val Accuracy: {val_acc:.2f}%")

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, 'best_model.pth')
            print("✓ Saved best model")

#### Run training loop

In [9]:
stats = split_pointing_data(
    data_dir="/content/data_1",
    output_dir="/content/dataset",
    train_ratio=0.85,
    val_ratio=0.15,
    test_ratio=0.0
)

# Configuration
DATA_DIR = "/content/dataset"  # Directory containing .jpg, .npy, .txt files
TRAIN_DIR = DATA_DIR + "/train"  # Training data directory
VAL_DIR = DATA_DIR + "/val"  # Validation data directory
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

# Create datasets
train_dataset = PointingDataset(TRAIN_DIR)
val_dataset = PointingDataset(VAL_DIR)

print(f"\nDataset info:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

# Create data loaders with custom collate function
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_pointing_batch
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_pointing_batch
)

model = BaseCNN()

# Or use YOLO model:
# model = create_yolo_pointing_net('small', input_height=1080, input_width=1920)

print(f"\nModel: {model.__class__.__name__}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train
train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, device=DEVICE)


Found 23 samples
✓ train: 19 samples
✓ val: 3 samples
✓ test: 1 samples
Using device: cuda
Found 19 samples in /content/dataset/train
Found 3 samples in /content/dataset/val

Dataset info:
  Training samples: 19
  Validation samples: 3

Model: BaseCNN
Total parameters: 2,004,519

Epoch 1/50
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Train Loss: 1.8399 (Conf: 0.7074, Vec: 1.1325)
Val Loss: 1.7365 (Conf: 0.7174, Vec: 1.0191)
Val Accuracy: 0.00%
✓ Saved best model

Epoch 2/50
--------------------------------------------------
Train Loss: 1.6521 (Conf: 0.6659, Vec: 0.9862)
Val Loss: 1.7090 (Conf: 0.7128, Vec: 0.9962)
Val Accuracy: 0.00%
✓ Saved best model

Epoch 3/50
--------------------------------------------------
Train Loss: 1.5582 (Conf: 0.6270, Vec: 0.9312)
Val Loss: 1.6537 (Conf: 0.6976, Vec: 0.9561)
Val Accuracy: 0.00%
✓ Saved best model

Epoch 4/50
--------------------------------------------------
Train Loss: 1.7796 (Conf: 0.5873, Vec: 1.1924)
Val Loss: 1.5732 (Conf: 0.6741, Vec: 0.8990)
Val Accuracy: 100.00%
✓ Saved best model

Epoch 5/50
--------------------------------------------------
Train Loss: 1.2117 (Conf: 0.5225, Vec: 0.6892)
Val Loss: 1.4498 (Conf: 0.6362, Vec: 0.8136)
Val Accuracy: 100.00%
✓ Saved best model

Epoch 6/50
--------------------------------------------------
Train Loss: 1.0486 (Conf: 